This file imports an EXCEL model (here template example from BW25 sharepoint folder) and matches it with the background database in the project. For this repository we are currently working with ecoinvent310clca

After successfully importing a database to a Brightway project, the database is stored in the project and doesn't have to be imported every time we open the repository. Unless we make changes to the background database of course

# EXCEL Model importer

#### 1. Import packages

In [2]:
# basic imports from brightway
import bw2analyzer as ba
import bw2calc as bc
import bw2data as bd
from bw2data import databases
import bw2io as bi
from bw2io import ExcelImporter
from bw2io.importers import SingleOutputEcospold2Importer
import bw2analyzer as bwa
from bw2data import methods
import argparse
import bw2data as bd
import os

# other relevant packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

13:51:02+0200 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


#### 2. Set project and see databases
The databases are required here, because the Excel database we want to import in this script needs to be matched to these.

In [3]:
# define a project where we install the databases and work in this script
bd.projects.set_current('LCA_Toolbox')

In [4]:
bd.databases

Databases dictionary with 8 object(s):
	BONSAI_V2.1.6
	BONSAI_V2.1.6 biosphere
	bafu
	biosphere3
	ecoinvent-3.10-biosphere
	ecoinvent-3.12-biosphere
	ecoinvent-3.12-consequential
	template-consequential

#### 3. Import the EXCEL file containing the model 
The file import is easy, we just use the Excel importer function from bw2io as can be seen below.

In [5]:
# Get user's home directory (C:\Users\USERNAME)
home = os.path.expanduser("~")

In [6]:
# Here you need to change the path to your local path where you have stored the excel file, this can also be on your sharepoint. 

BW_LCI_importer_template = bi.ExcelImporter(os.path.join(
    home,
    "OneDrive - 2.-0 LCA Consultants ApS",  # This stays the same for all users
    "Intranet - Brightway2",
    "excel_model_example",
    "BW_LCI_importer_template.xlsx"
))

Extracted 1 worksheets in 0.09 seconds


#### 4. Importing means matching with background databases and writing the database
The part that is a bit trickier to understand is how the matching with the relevant background databases works. In essence, you need to match the newly imported databases with all background databases used in the model.

In [8]:
BW_LCI_importer_template.apply_strategies()
BW_LCI_importer_template.match_database(fields=['name','location'])
BW_LCI_importer_template.match_database(db_name='ecoinvent-3.12-consequential', fields=['name','location','unit'])
BW_LCI_importer_template.match_database(db_name='ecoinvent-3.12-biosphere', fields=['name','categories','unit'])
BW_LCI_importer_template.match_database(db_name='biosphere3', fields=['name','categories','unit'])

Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applying strategy: convert_activity_parameters_to_list
Applied 14 strategies in 1.74 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields


In [9]:
# check foreground imports for unlinked processes
pd.DataFrame(BW_LCI_importer_template.unlinked)

""


In [11]:
# if there are unlinked exchanges, this helps localizing them in order to fix them
list(BW_LCI_importer_template.unlinked)


[]

In [12]:
BW_LCI_importer_template.write_database()

13:53:16+0200 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|██████████| 1/1 [00:00<?, ?it/s]


13:53:16+0200 [info     ] Vacuuming database            
Created database: template_consequential


#### 5. Store data in a dataframe so it can be analysed in Brightway 2.5
This step allows you to analyse the imported database in the Brightway framework.

In [13]:
LCI_template = bd.Database('BW_LCI_importer_template')


In [18]:
bd.databases

Databases dictionary with 8 object(s):
	BONSAI_V2.1.6
	BONSAI_V2.1.6 biosphere
	bafu
	biosphere3
	ecoinvent-3.10-biosphere
	ecoinvent-3.12-biosphere
	ecoinvent-3.12-consequential
	template_consequential

In [19]:
ei_bio = bd.Database('ecoinvent-3.12-biosphere')
template_consequential = bd.Database('template_consequential')
el_bio3 = bd.Database('biosphere3')
ei_clca = bd.Database('ecoinvent-3.12-consequential')
bafu = bd.Database('bafu')

In [20]:
template_consequential = [activity.as_dict() for activity in template_consequential]


In [21]:
df_template_consequential = pd.DataFrame(template_consequential)


In [22]:
from IPython.display import display
display(df_template_consequential)

,comment,location,production amount,reference product,unit,name,worksheet name,database,code,type,id
0,This dataset represents production of porcelai...,RER,1,porcelain ceramics,kilogram,production of porcelain ceramics,BW_LCI_import,template_consequential,18cf9c1477062c6c32dd5fb61d8a3a06,processwithreferenceproduct,310745904129060864
